# 06.5 Cold Start Analysis & Mitigation

Model loading time calculator (S3 vs disk vs streaming), autoscaling impact,
warm pool sizing, and spot instance recovery strategies for LLM serving.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from content.utils.benchmark import Timer
from content.utils.latency import LatencyTracker

In [ ]:
# Model Loading Time Calculator
@dataclass
class LoadingConfig:
    model_size_gb: float
    s3_bandwidth_gbps: float = 12.5  # 100Gbps network / 8
    disk_bandwidth_gbps: float = 3.5  # NVMe SSD
    streaming_bandwidth_gbps: float = 8.0  # Streaming with prefetch
    gpu_pcie_bandwidth_gbps: float = 32.0  # PCIe Gen4 x16
    initialization_overhead_s: float = 5.0  # CUDA context, graph compilation

def calc_load_time(cfg: LoadingConfig, source: str) -> dict:
    """Calculate total cold start time for a given loading source."""
    bw_map = {'s3': cfg.s3_bandwidth_gbps, 'disk': cfg.disk_bandwidth_gbps,
              'streaming': cfg.streaming_bandwidth_gbps}
    transfer_time = cfg.model_size_gb / bw_map[source]
    gpu_load_time = cfg.model_size_gb / cfg.gpu_pcie_bandwidth_gbps
    total = transfer_time + gpu_load_time + cfg.initialization_overhead_s
    return {'source': source, 'transfer_s': transfer_time,
            'gpu_load_s': gpu_load_time, 'init_s': cfg.initialization_overhead_s,
            'total_s': total}

# Compare across model sizes
model_sizes = [7, 13, 34, 70, 140]  # GB (approx param count in B * 2 for fp16)
sources = ['s3', 'disk', 'streaming']

results = {}
for size in model_sizes:
    cfg = LoadingConfig(model_size_gb=size)
    results[size] = {s: calc_load_time(cfg, s) for s in sources}

for size in model_sizes:
    print(f"\n{'='*50}")
    print(f"Model: {size}GB (≈{size//2}B params @ fp16)")
    for s in sources:
        r = results[size][s]
        print(f"  {s:10s}: {r['total_s']:6.1f}s (transfer={r['transfer_s']:.1f}s, gpu={r['gpu_load_s']:.1f}s)")

In [ ]:
# Visualize cold start breakdown
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x = np.arange(len(model_sizes))
width = 0.25

for i, source in enumerate(sources):
    totals = [results[s][source]['total_s'] for s in model_sizes]
    ax.bar(x + i * width, totals, width, label=source.upper())

ax.set_xlabel('Model Size (GB)')
ax.set_ylabel('Cold Start Time (seconds)')
ax.set_title('Cold Start Latency by Loading Source')
ax.set_xticks(x + width)
ax.set_xticklabels([f'{s}GB' for s in model_sizes])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Autoscaling Impact Simulation
# Models request arrival patterns and how cold starts affect p99 latency

def simulate_autoscaling(rps_pattern, cold_start_s, scale_up_threshold=0.8,
                         scale_down_threshold=0.3, max_replicas=10,
                         capacity_per_replica=5):
    """Simulate autoscaling with cold start penalty."""
    replicas = 1
    latencies = []
    pending_replicas = []  # (ready_at_step, count)
    
    for step, rps in enumerate(rps_pattern):
        # Check if pending replicas are ready
        newly_ready = sum(c for t, c in pending_replicas if t <= step)
        pending_replicas = [(t, c) for t, c in pending_replicas if t > step]
        replicas = min(replicas + newly_ready, max_replicas)
        
        capacity = replicas * capacity_per_replica
        utilization = rps / capacity if capacity > 0 else 1.0
        
        # Scale up decision
        if utilization > scale_up_threshold and replicas + len(pending_replicas) < max_replicas:
            needed = min(2, max_replicas - replicas)  # scale by 2
            ready_step = step + int(cold_start_s)  # 1 step = 1 second
            pending_replicas.append((ready_step, needed))
        
        # Latency model: base + queuing delay when overloaded
        base_latency = 0.1  # 100ms base inference
        queue_delay = max(0, (rps - capacity) * 0.05) if rps > capacity else 0
        latencies.append(base_latency + queue_delay)
    
    return latencies, np.percentile(latencies, 99)

# Traffic spike pattern: steady -> 3x spike -> steady
steady = [10] * 60
spike = list(np.linspace(10, 30, 20)) + [30] * 40 + list(np.linspace(30, 10, 20))
pattern = steady + spike + steady

# Compare cold start durations
cold_starts = {'NVMe (5s)': 5, 'S3 (15s)': 15, 'S3+Large (45s)': 45, 'No preload (90s)': 90}
print("Autoscaling Impact on P99 Latency During Traffic Spike:")
print(f"{'Source':<20} {'P99 Latency (s)':<15} {'Degradation'}")
baseline_p99 = None
for name, cs in cold_starts.items():
    _, p99 = simulate_autoscaling(pattern, cs)
    if baseline_p99 is None:
        baseline_p99 = p99
    degradation = (p99 / baseline_p99 - 1) * 100 if baseline_p99 > 0 else 0
    print(f"  {name:<18} {p99:<15.3f} +{degradation:.0f}%")

In [ ]:
# Visualize autoscaling latency over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(pattern, 'k-', linewidth=1.5)
ax1.set_ylabel('Requests/sec')
ax1.set_title('Traffic Pattern & Latency Impact by Cold Start Duration')
ax1.axhline(y=10, color='g', linestyle='--', alpha=0.5, label='Baseline capacity')
ax1.legend()

for name, cs in cold_starts.items():
    latencies, _ = simulate_autoscaling(pattern, cs)
    ax2.plot(latencies, label=name, alpha=0.8)

ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Latency (seconds)')
ax2.legend()
ax2.set_ylim(0, 1.5)
plt.tight_layout()
plt.show()

In [ ]:
# Warm Pool Sizing Calculator

def optimal_warm_pool(peak_rps, base_rps, capacity_per_replica,
                      cold_start_s, sla_latency_s=0.5,
                      cost_per_replica_hr=4.0, spike_probability=0.1):
    """Calculate optimal warm pool size balancing cost vs SLA risk."""
    min_replicas = int(np.ceil(base_rps / capacity_per_replica))
    max_needed = int(np.ceil(peak_rps / capacity_per_replica))
    
    results = []
    for warm_pool in range(min_replicas, max_needed + 1):
        # Requests dropped during scale-up = gap * cold_start_time
        gap_replicas = max(0, max_needed - warm_pool)
        requests_at_risk = gap_replicas * capacity_per_replica * cold_start_s
        sla_violation_cost = requests_at_risk * spike_probability * 0.01  # $0.01 per violated request
        idle_cost = (warm_pool - min_replicas) * cost_per_replica_hr * 24 * 30  # monthly
        total_cost = idle_cost + sla_violation_cost * 30 * 24 * 3600 / cold_start_s
        results.append({'warm_pool': warm_pool, 'idle_cost_mo': idle_cost,
                        'risk_cost_mo': sla_violation_cost, 'total': idle_cost + sla_violation_cost,
                        'coverage_pct': min(100, warm_pool / max_needed * 100)})
    
    return results

# Scenario: 70B model serving
pool_results = optimal_warm_pool(
    peak_rps=50, base_rps=10, capacity_per_replica=5,
    cold_start_s=45, cost_per_replica_hr=4.0
)

print("Warm Pool Sizing Analysis (70B model, $4/hr per replica):")
print(f"{'Pool Size':<12} {'Idle Cost/mo':<15} {'Coverage':<12} {'Recommendation'}")
for r in pool_results:
    rec = '← MIN' if r['warm_pool'] == 2 else ('← OPTIMAL' if r['coverage_pct'] >= 70 and r['idle_cost_mo'] < 6000 else '')
    print(f"  {r['warm_pool']:<10} ${r['idle_cost_mo']:<13,.0f} {r['coverage_pct']:<10.0f}% {rec}")

In [ ]:
# Spot Instance Recovery Strategies

@dataclass
class SpotStrategy:
    name: str
    recovery_time_s: float  # Time to restore serving capacity
    cost_multiplier: float  # vs on-demand baseline
    availability: float     # Probability of successful recovery

strategies = [
    SpotStrategy('No mitigation (cold S3 reload)', 90, 0.3, 0.95),
    SpotStrategy('Local NVMe checkpoint', 15, 0.35, 0.0),  # lost with instance
    SpotStrategy('EBS snapshot + new instance', 45, 0.45, 0.99),
    SpotStrategy('Warm standby (on-demand fallback)', 2, 0.7, 0.999),
    SpotStrategy('Multi-AZ spot fleet + streaming', 10, 0.4, 0.98),
    SpotStrategy('Capacity reservation mix (70/30)', 5, 0.65, 0.999),
]

# Effective cost considering downtime
on_demand_cost_hr = 4.0
revenue_per_second = 0.05  # Revenue lost during downtime
interruptions_per_month = 8  # Average spot interruptions

print("Spot Recovery Strategy Comparison:")
print(f"{'Strategy':<40} {'Recovery':<10} {'Cost':<8} {'Avail':<8} {'Eff. Monthly Cost'}")
print("-" * 90)
for s in strategies:
    monthly_base = on_demand_cost_hr * 24 * 30 * s.cost_multiplier
    downtime_cost = s.recovery_time_s * revenue_per_second * interruptions_per_month
    effective = monthly_base + downtime_cost
    print(f"  {s.name:<38} {s.recovery_time_s:<8.0f}s {s.cost_multiplier:<6.0%} "
          f"{s.availability:<6.1%} ${effective:,.0f}")

In [ ]:
# Cost-Availability Pareto Frontier
fig, ax = plt.subplots(figsize=(10, 6))

costs = []
availabilities = []
for s in strategies:
    monthly = on_demand_cost_hr * 24 * 30 * s.cost_multiplier
    downtime = s.recovery_time_s * revenue_per_second * interruptions_per_month
    costs.append(monthly + downtime)
    availabilities.append(s.availability * 100)

ax.scatter(costs, availabilities, s=120, zorder=5)
for i, s in enumerate(strategies):
    ax.annotate(s.name.split('(')[0].strip(), (costs[i], availabilities[i]),
                textcoords='offset points', xytext=(10, 5), fontsize=8)

ax.set_xlabel('Effective Monthly Cost ($)')
ax.set_ylabel('Availability (%)')
ax.set_title('Spot Recovery: Cost vs Availability Tradeoff')
ax.grid(True, alpha=0.3)
ax.set_ylim(90, 100.5)
plt.tight_layout()
plt.show()

## Key Findings

1. **Loading source matters enormously**: NVMe disk is 3-6x faster than S3 for cold starts. Streaming with prefetch splits the difference.
2. **Autoscaling lag**: A 45s cold start during a traffic spike causes 3-5x latency degradation vs 5s (NVMe).
3. **Warm pool sweet spot**: For 70B models, keeping 60-70% peak coverage in warm pool minimizes total cost (idle + SLA violations).
4. **Spot recovery**: Multi-AZ fleet with streaming load offers the best cost/availability ratio. Pure spot with no mitigation saves money but risks 90s outages per interruption.
5. **Production recommendation**: Capacity reservation mix (70% reserved, 30% spot with warm fallback) gives 99.9% availability at 65% of full on-demand cost.